Compressing Large Language Models (LLMs) is crucial for deploying them on resource-constrained devices and reducing inference latency. Several methods exist, each with its trade-offs in terms of compression ratio and performance. Here's an overview of common techniques, each applied to a real model so you can measure the trade-off:

**1. Quantization:**

* **Concept:** Reduces the precision of model weights and activations (e.g., from 32-bit floating-point to 8-bit integer).
* **Methods:**
    * Post-Training Quantization (PTQ): Quantizes a pre-trained model without further training.
    * Quantization-Aware Training (QAT): Incorporates quantization into the training process.

Quantization is covered in depth (size, perplexity, latency) in `8_W_QuantizedLLM.ipynb`; the helper below uses the
current PyTorch API (`torchao`). Requires `pip install torch torchao transformers scikit-learn`.

In [1]:
import io
import copy
import math
import warnings
import pandas as pd
import torch
import torch.nn as nn
from torchao.quantization import quantize_, Int8WeightOnlyConfig

torch.manual_seed(0)
device = torch.device("cpu")

def quantize_model(model):
    """Post-training quantization: int8 weights for every nn.Linear (in place)."""
    quantize_(model, Int8WeightOnlyConfig())
    return model

# Quantization-aware training (QAT) inserts "fake quantization" during fine-tuning so the model learns to tolerate
# int8 rounding; see torchao.quantization.qat (IntXQuantizationAwareTrainingConfig) for the current API.

## Shared setup: model and perplexity
`facebook/opt-125m` is used because its layers are `nn.Linear`; GPT-2 uses a custom `Conv1D` class, so code that
loops over `nn.Linear` modules would silently skip almost every layer.

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "facebook/opt-125m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32).to(device).eval()
texts = [p.strip() for p in open("Data/llm_eval_text.txt").read().split("\n\n") if p.strip()]

def calculate_perplexity(model):
    total_nll, total_tokens = 0.0, 0
    with torch.inference_mode():
        for text in texts:
            ids = tokenizer(text, return_tensors="pt").input_ids.to(device)
            n = ids.shape[1] - 1  # number of predicted tokens
            total_nll += model(ids, labels=ids).loss.item() * n
            total_tokens += n
    return math.exp(total_nll / total_tokens)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def nonzero_params(model):
    return sum(int((p != 0).sum()) for p in model.parameters())

def size_mb(model):
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.tell() / 1e6

# Compress the transformer layers only; lm_head is tied to the input embedding table
def compressible_linears(model):
    return [(name, m) for name, m in model.named_modules() if isinstance(m, nn.Linear) and name != "lm_head"]

baseline_ppl = calculate_perplexity(base_model)
print(f"Baseline: {count_params(base_model) / 1e6:.1f}M params, {size_mb(base_model):.0f} MB, perplexity {baseline_ppl:.2f}")
print(f"Compressible nn.Linear layers: {len(compressible_linears(base_model))}")

Baseline: 125.2M params, 501 MB, perplexity 31.66
Compressible nn.Linear layers: 72


**2. Pruning:**

* **Concept:** Removes redundant or less important connections (weights) from the model.
* **Methods:**
    * Weight (unstructured) pruning: Sets individual weights to zero.
    * Neuron pruning: Removes entire neurons.
    * Structural Pruning: removes larger structures like attention heads.

Unstructured pruning zeroes weights but the tensors keep their shape, so the file size and speed do **not** change
unless you store them in a sparse format or run on hardware with sparse kernels. Structured pruning actually removes
rows/heads and shrinks the matrices.

In [3]:
import torch.nn.utils.prune as prune

def prune_model(model, amount=0.2):
    """Unstructured L1 pruning of every compressible nn.Linear."""
    for _, module in compressible_linears(model):
        prune.l1_unstructured(module, name='weight', amount=amount)
        prune.remove(module, 'weight')  # makes the changes permanent
    return model

rows = []
for amount in [0.0, 0.1, 0.3, 0.5, 0.7]:
    pruned = prune_model(copy.deepcopy(base_model), amount)
    rows.append({"pruned_fraction": amount,
                 "nonzero_params_M": nonzero_params(pruned) / 1e6,
                 "size_MB": size_mb(pruned),
                 "perplexity": calculate_perplexity(pruned)})
    del pruned
pd.DataFrame(rows).round(2)

,pruned_fraction,nonzero_params_M,size_MB,perplexity
0,0.0,125.24,501.03,31.66
1,0.1,116.75,501.03,32.42
2,0.3,99.76,501.03,39.56
3,0.5,82.77,501.03,136.58
4,0.7,65.78,501.03,1836.91


Perplexity stays close to the baseline at 10% sparsity, then degrades quickly, and the file size never changes. In practice pruned models are
**fine-tuned** afterwards to recover accuracy, and LLM-specific methods (e.g. SparseGPT, Wanda) choose which weights to
remove more carefully than plain magnitude.

**3. Low-Rank Factorization:**

* **Concept:** Decomposes a weight matrix W (out x in) into two thin matrices A (out x r) and B (r x in), reducing
  parameters from out*in to r*(out+in).
* **Methods:**
    * Singular Value Decomposition (SVD): keep the top-r singular values - the best rank-r approximation of W.
    * Tensor decomposition (e.g. with Tensorly) for higher-order weights.
* The same idea underlies **LoRA**, which trains a low-rank *update* instead of compressing the original weights.

In [4]:
def factorize_linear(linear, rank):
    """Replace one nn.Linear with two smaller ones using truncated SVD."""
    U, S, Vh = torch.linalg.svd(linear.weight.data, full_matrices=False)
    first = nn.Linear(linear.in_features, rank, bias=False)
    second = nn.Linear(rank, linear.out_features, bias=linear.bias is not None)
    first.weight.data = (torch.diag(S[:rank].sqrt()) @ Vh[:rank]).contiguous()
    second.weight.data = (U[:, :rank] @ torch.diag(S[:rank].sqrt())).contiguous()
    if linear.bias is not None:
        second.bias.data = linear.bias.data.clone()
    return nn.Sequential(first, second)

def low_rank_model(model, rank, targets=("fc1", "fc2")):
    """Factorize the feed-forward (MLP) layers of every decoder block."""
    for layer in model.model.decoder.layers:
        for name in targets:
            setattr(layer, name, factorize_linear(getattr(layer, name), rank))
    return model

rows = [{"rank": "full (768 x 3072)", "params_M": count_params(base_model) / 1e6, "perplexity": baseline_ppl}]
for rank in [512, 384, 256]:
    lr_model = low_rank_model(copy.deepcopy(base_model), rank).eval()
    rows.append({"rank": rank, "params_M": count_params(lr_model) / 1e6, "perplexity": calculate_perplexity(lr_model)})
    del lr_model
pd.DataFrame(rows).round(2)

,rank,params_M,perplexity
0,full (768 x 3072),125.24,31.66
1,512,115.80,111.10
2,384,104.01,365.31
3,256,92.21,4309.08


Even keeping rank 512 of 768 sends perplexity far above the baseline while saving under 10% of the parameters:
trained LLM weight matrices are not close to low-rank, so plain SVD compression is a poor trade on its own. In practice it is
followed by fine-tuning, uses activation-aware variants (e.g. ASVD, SVD-LLM), or is applied only to layers whose singular
values fall off quickly.

**4. Knowledge Distillation:**

* **Concept:** Trains a smaller "student" model to mimic the behavior of a larger "teacher" model.
* **Methods:**
    * Soft labels: Using the teacher's output probabilities (soft labels) as targets for the student.
    * Intermediate feature matching: Matching the feature representations of the teacher and student models.

Distilling an LLM takes far more compute than a notebook allows, so the mechanics are shown on a small image
classifier (8x8 handwritten digits).

Distillation pays off most when labeled data is scarce but **unlabeled inputs are plentiful** - exactly the LLM
situation, where a student learns from the teacher's outputs on large amounts of raw text. Here the teacher is already
trained, and the student gets only 100 labeled examples plus the remaining training images *without* labels:

* **Student (labels only)** trains on the 100 labeled examples.
* **Student (distilled)** uses the same 100 labels plus the teacher's soft labels on all training images.

In [5]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

X, y = load_digits(return_X_y=True)
X = torch.tensor(X / 16.0, dtype=torch.float32)
y = torch.tensor(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

def mlp(hidden):
    layers, size = [], 64
    for h in hidden:
        layers += [nn.Linear(size, h), nn.ReLU()]
        size = h
    return nn.Sequential(*layers, nn.Linear(size, 10))

def accuracy(model):
    model.eval()
    with torch.no_grad():
        return (model(X_test).argmax(1) == y_test).float().mean().item()

def train_supervised(model, train_loader, epochs):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        model.train()
        for data, target in train_loader:
            optimizer.zero_grad()
            nn.functional.cross_entropy(model(data), target).backward()
            optimizer.step()
    return model

# Teacher: a large network trained on the full labeled training set (stands in for a big pre-trained model)
torch.manual_seed(0)
teacher = train_supervised(mlp([512, 512]), DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True), epochs=30)

# Student data: 100 labeled examples; every other training image is unlabeled (label = -1)
n_labeled = 100
labeled_idx = torch.randperm(len(X_train), generator=torch.Generator().manual_seed(0))[:n_labeled]
y_partial = torch.full_like(y_train, -1)
y_partial[labeled_idx] = y_train[labeled_idx]
labeled_loader = DataLoader(TensorDataset(X_train[labeled_idx], y_train[labeled_idx]), batch_size=32, shuffle=True)
transfer_loader = DataLoader(TensorDataset(X_train, y_partial), batch_size=32, shuffle=True)

In [6]:
def train_student(student_model, teacher_model, train_loader, optimizer, alpha=0.9, temperature=4.0, epochs=1):
    """Trains a student model using knowledge distillation. Targets of -1 mark unlabeled examples."""
    teacher_model.eval()
    for _ in range(epochs):
        student_model.train()
        for data, target in train_loader:
            optimizer.zero_grad()
            student_output = student_model(data)
            with torch.no_grad():  # the teacher is fixed; don't build a graph for it
                teacher_output = teacher_model(data)

            soft_targets = torch.softmax(teacher_output / temperature, dim=1)
            soft_prob = torch.log_softmax(student_output / temperature, dim=1)
            # T^2 keeps the gradient scale comparable to the hard loss
            distillation_loss = nn.functional.kl_div(soft_prob, soft_targets, reduction='batchmean') * (temperature ** 2)

            labeled = target >= 0  # hard loss uses the true labels, where we have them
            hard_loss = nn.functional.cross_entropy(student_output[labeled], target[labeled]) if labeled.any() else 0.0

            loss = alpha * distillation_loss + (1 - alpha) * hard_loss
            loss.backward()
            optimizer.step()
    return student_model

student_hidden = [32]

torch.manual_seed(1)
student_alone = train_supervised(mlp(student_hidden), labeled_loader, epochs=100)  # more epochs: only 4 batches per epoch

torch.manual_seed(1)
student = mlp(student_hidden)
student_distilled = train_student(student, teacher, transfer_loader, torch.optim.Adam(student.parameters(), lr=1e-3), epochs=30)

pd.DataFrame([
    {"model": "teacher (all labels)", "params": count_params(teacher), "test_accuracy": accuracy(teacher)},
    {"model": f"student ({n_labeled} labels only)", "params": count_params(student_alone), "test_accuracy": accuracy(student_alone)},
    {"model": f"student ({n_labeled} labels + teacher)", "params": count_params(student_distilled), "test_accuracy": accuracy(student_distilled)},
]).set_index("model").round(4)

,params,test_accuracy
model,,
teacher (all labels),301066,0.9741
student (100 labels only),2410,0.8722
student (100 labels + teacher),2410,0.9463


The distilled student is ~125x smaller than the teacher yet recovers most of its accuracy, because the teacher's soft
labels turn unlabeled images into training signal - and say *how* similar classes are ("this 7 looks a bit like a 1").
With plentiful clean labels the advantage shrinks; distillation matters most when the teacher knows something the labeled data doesn't.

**Important Notes:**

* Compression techniques often involve trade-offs between model size, performance, and accuracy - measure all three.
* The best compression method depends on the specific LLM architecture, task, and hardware constraints.
* Techniques combine: e.g. distill a smaller model, then prune and quantize it.
* Libraries such as `torchao`, Hugging Face `transformers` / `optimum`, and `llm-compressor` provide production implementations.
* When using quantization, make sure your hardware supports the datatype you are quantizing to (for instance, int8).
* When using pruning or low-rank factorization, fine-tune the compressed model to recover lost accuracy.

<div class="md-recitation">
  Sources
  <ol>
  <li><a href="https://github.com/HeadMelting/AI-Basis">https://github.com/HeadMelting/AI-Basis</a></li>
  <li><a href="https://tech.jxpress.net/entry/2021/08/26/170000">https://tech.jxpress.net/entry/2021/08/26/170000</a></li>
  </ol>
</div>